# import & API key 

In [1]:
import os
from dotenv import load_dotenv
print(load_dotenv())
print(os.getenv('OPENAI_API_KEY')[:20])

import os
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.runnables.passthrough import RunnablePassthrough

True
sk-proj-iKU13YeoxNgF


# RAG
Retrieval Augmented Generation (검색증강생성)

In [ ]:
# 사전학습된 모델은 이미 많은 데이터를 통해 학습한 상태이긴 하나..
# 개인 DB 나 회사내 문서  와 같이 'private 한 데이터' 들에는 접근할수 없다
# 그래서 RAG 를 사용하여 LLM !

![](https://www.dropbox.com/scl/fi/j0b5nef451hxzme9hnv6j/chain_stuff.png?rlkey=5y8i9gtmmxya9jlbx2o51cuvd&st=s98d8lx7&dl=1)

In [ ]:
# 1. Retrieval 단계
# private 으로부터 제공된 data 를 사용하거나 탐색함으로써
# language model 의 능력을 더 '확장(augment)'

# 2. Augmented Generation
# Model 로 하여금 '우리가 보낸 문서 data 만'을 가지고 답변하도록 할수도 있다.
# (경우에 따라, 우리 문서가 더 최신 data 일수도 있기 때문이다)
# 이를 통해 Model 이 과거에 학습한 data 를 참조하지 않게도 할수 있다.

In [ ]:
# RAG 는 특정 라이브러리나 프레임워크 이름이 아니라
# 위와 같은 작업을 하는 '기법'을 일반적으로 통칭하는 용어

# RAG 를 수행하는 방법은 굉~장히 많고 다양.

![](https://www.dropbox.com/scl/fi/tcucb37mbys2ptyddc2i8/chain_map_reduce.png?rlkey=6kygllnh8ly927h79zecs9m10&st=113d42y3&dl=1)

![](https://www.dropbox.com/scl/fi/x5ihbsscafhtafve5bctw/chain_map_rerank.png?rlkey=hkw0mxbnjoggen23pzmz67s84&st=nf5smeno&dl=1)

![](https://www.dropbox.com/scl/fi/77vb082uo1g4n2n4muqwn/chain_refine.png?rlkey=uywjls0hzefaw1ubuks6le47x&st=sovw6cd1&dl=1)

In [ ]:
# 어떤 방식으로 RAG 를 구현할른지는

# - 우리가 얼마나 많은 문서들을 가지고 있는지
# - 우리가 얼마나 많은 비용으로 운영할지 (어떤 모델, 가용한 token 개수등..)

# 등에 따라 결정될 문제다.

# Retrieve 란
![](https://www.dropbox.com/scl/fi/7z3te93sg2rzvtwl6jq9q/retreive.png?rlkey=cgy5eh93doyb09263zgiswnp2&st=hcw03qsu&dl=1)

In [2]:
# RAG 의 첫번째 단계인 Retrieval 의 일반적인 과정
# - data source 에서 데이터 load
# - 데이터는 split 하면서 transform
# - transform 한 데이터를 embed.
# - embed 된 데이터를 store 에 저장.
# - 검색(질의) 가 입력되면 store 에서 관련 문서들을 retrieve!

# 🟦 Data Loaders
- https://docs.langchain.com/oss/python/integrations/document_loaders  (2026.6 현재)

## 파일 준비

In [ ]:
# 아래와 같이 파일들을 준비합니다

# 출처는  조지오웰의 소설 '1984' Part1 Chapter1
#  http://www.george-orwell.org/1984/0.html

# 너무 길거나, 너무 짧지 않으면 좋습니다
# 파일이 너무 길면 나중에 임베딩 과정에서 비용지출이 발생.

# 다운로드 링크
# https://www.dropbox.com/scl/fi/ppid6hk7bwqxc0xrv65oc/files.zip?rlkey=df5d411n1fnaht2wlwv0o71wt&st=ddd52gw3&dl=1


In [3]:
llm = ChatOpenAI(
    model='gpt-4o',
    temperature=0.1,
)

base_path = r'D:\MCP2607\LangWork\dataset'

## Text Loader

In [5]:
import os
from langchain_core.documents import Document

In [6]:
file_path = os.path.join(base_path, 'files', 'chapter_one.txt')

with open(file_path, encoding='utf-8') as f:
    docs = [Document(page_content=f.read(), metadata={"source": file_path})]

docs



[Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hat

In [7]:
len(docs)

1

## BaseLoader
DataLoader 커스터마이징

In [8]:
from typing import Iterator
from langchain_core.document_loaders import BaseLoader

class TextLoader(BaseLoader):
    def __init__(self, file_path: str, encoding: str = 'utf-8'):
        self.file_path = file_path
        self.encoding = encoding

    # lazy_load 만 구현하면 BaseLoader 가 load() 를 자동으로 제공해준다.
    def lazy_load(self) -> Iterator[Document]:
        with open(self.file_path, encoding=self.encoding) as f:
            yield Document(page_content=f.read(), metadata={"source": self.file_path})


In [9]:
loader = TextLoader(file_path)
docs = loader.load()
docs

[Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hat

## PDF Loader

In [10]:
from pypdf import PdfReader

In [11]:
file_path = os.path.join(base_path, 'files', 'chapter_one.pdf')
reader = PdfReader(file_path)

In [12]:
docs = [
    Document(
        page_content=page.extract_text() or "",
        metadata={"source": file_path, "page": i, "total_pages": len(reader.pages)}
    )
    for i, page in enumerate(reader.pages)
]

print(len(docs))
docs

15


[Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.pdf', 'page': 0, 'total_pages': 15}, page_content='Part 1, Chapter 1 \n \n \nPart One \n \n \n1 \nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his \nchin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through \nthe glass doors of Victory Mansions, though not quickly enough to prevent a swirl of \ngritty dust from entering along with him. \n \nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured \nposter, too large for indoor display, had been tacked to the wall. It depicted simply an \nenormous face, more than a metre wide: the face of a man of about forty-five, with a \nheavy black moustache and ruggedly handsome features. Winston made for the stairs. It \nwas no use trying the lift. Even at the best of times it was seldom working, and at \npresent the electric current was cut off during daylight hours

## UnstructuredLoader

서로 다른 타입의 문서를 읽어오기 위해 각각의 DataLoader 를 사용하기 보다 UnstructuredLoader 라는 것도 사용해볼수 있다. -> 꽤 다양한 포맷의 파일을 읽어올 수 있다

Unstructured 연동은 langchain-unstructured 공식패키지가 제공되고, UnstructuredLoader클래스 제공. 

기본적으로 로컬의 unstructured 라이브러리로 파일을 분석하고 파일 형식도 자동으로 감지


In [13]:
from langchain_unstructured import UnstructuredLoader

In [14]:
# pdf
loader = UnstructuredLoader(os.path.join(base_path, 'files', 'chapter_one.pdf'))
docs = loader.load()
print(len(docs))
docs[:5]

INFO: pikepdf C++ to Python logger bridge initialized
INFO: Downloading spaCy model en_core_web_sm 3.8.0 …
INFO: Installed en_core_web_sm 3.8.0


445


[Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.pdf', 'coordinates': {'points': ((72.024, 90.428), (72.024, 101.46799999999996), (165.54608, 101.46799999999996), (165.54608, 90.428)), 'system': 'PixelSpace', 'layout_width': 595.32, 'layout_height': 841.92}, 'file_directory': 'D:\\MCP2607\\LangWork\\dataset\\files', 'filename': 'chapter_one.pdf', 'last_modified': '2025-01-30T23:19:00', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/pdf', 'category': 'Title', 'element_id': '5c61737bbad75c4502a9315f7969b29e'}, page_content='Part 1, Chapter 1'),
 Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.pdf', 'coordinates': {'points': ((72.024, 163.50799999999992), (72.024, 174.5479999999999), (120.18607999999999, 174.5479999999999), (120.18607999999999, 163.50799999999992)), 'system': 'PixelSpace', 'layout_width': 595.32, 'layout_height': 841.92}, 'file_directory': 'D:\\MCP2607\\LangWork\\dataset\\files', 'filename

In [ ]:
# 쪼개는 기준은 문서의 "구조 요소(element)" 단위입니다. 글자 수나 토큰 수로 자르는 게 아니라, 
# unstructured 라이브러리가 PDF를 파싱해서 찾아낸 제목, 문단, 리스트 항목, 표, 헤더/푸터 같은 
# 요소 하나하나가 각각 Document 1개가 됩니다. 그래서 445개처럼 많이 나옵니다

In [15]:
# txt
loader = UnstructuredLoader(os.path.join(base_path, 'files', 'chapter_one.txt'))
docs = loader.load()
print(len(docs))
docs[:5]

# txt도 같은 원리입니다. 내부적으로 partition_text()가 호출되고, 텍스트를 줄바꿈 기준으로 잘라 각 조각을 하나의 element로 만듭니다. 그다음 각 조각을 Title, NarrativeText, ListItem 등으로 분류합니다.

# 쪼개지는 기준
# unstructured의 문단 분리 패턴은 \s*\n\s* 형태입니다. 빈 줄(\n\n)뿐 아니라 단순 줄바꿈 하나에서도 잘립니다. (버전에 따라 세부 동작이 조금 다를 수 있습니다.)
# 그래서 문장 중간에 줄바꿈이 들어간 txt(예: 한 줄 80자로 맞춘 파일)는 한 줄 한 줄이 Document가 됩니다.
# 한 조각이 너무 길면 max_partition(기본 1500자) 기준으로 추가로 나뉩니다.
# 짧고 문장부호가 없는 줄은 Title로 분류되는 경우가 많습니다.

58


[Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt', 'last_modified': '2025-01-30T23:18:24', 'languages': ['eng'], 'file_directory': 'D:\\MCP2607\\LangWork\\dataset\\files', 'filename': 'chapter_one.txt', 'filetype': 'text/plain', 'category': 'Title', 'element_id': '37876bee7fe989e27042aa191eedef0b'}, page_content='Part 1, Chapter 1'),
 Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt', 'last_modified': '2025-01-30T23:18:24', 'languages': ['eng'], 'file_directory': 'D:\\MCP2607\\LangWork\\dataset\\files', 'filename': 'chapter_one.txt', 'filetype': 'text/plain', 'category': 'Title', 'element_id': 'ed00f9cb74ff13d10b24cbd7b5607a22'}, page_content='Part One'),
 Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt', 'last_modified': '2025-01-30T23:18:24', 'languages': ['eng'], 'file_directory': 'D:\\MCP2607\\LangWork\\dataset\\files', 'filename': 'chapter_one.txt', 'filetype': 'text/plain

In [16]:
# docx

loader = UnstructuredLoader(os.path.join(base_path, 'files', 'chapter_one.docx'))
docs = loader.load()
print(len(docs))
docs[:5]

30


[Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.docx', 'category_depth': 0, 'emphasized_text_contents': ['Part 1, Chapter 1'], 'emphasized_text_tags': ['b'], 'file_directory': 'D:\\MCP2607\\LangWork\\dataset\\files', 'filename': 'chapter_one.docx', 'last_modified': '2025-01-30T23:18:37', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'UncategorizedText', 'element_id': '12511a37971ae1d93658752eb8428450'}, page_content='Part 1, Chapter 1'),
 Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.docx', 'category_depth': 0, 'file_directory': 'D:\\MCP2607\\LangWork\\dataset\\files', 'filename': 'chapter_one.docx', 'last_modified': '2025-01-30T23:18:37', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'NarrativeText', 'element_id': 'a04cbc9f79c4b5f9222

# 🟦 Splitter

## data 를 split 해야 되는 이유

In [ ]:
# loader.load() 의 리턴값은 Document 들의 list 다.

In [17]:
loader = TextLoader(os.path.join(base_path, 'files', 'chapter_one.txt'))
len(loader.load())

1

In [ ]:
# 특정 질문에 답해야 하기 위해서, 필요한 '파일의 일부분' 만들 전달해야 할 수도 있다.

#  그래서 문서를 쪼개두어야(split) 한다

# 가령: "Ministry of peace" 를 찾고자 한다면.
# 해당 키워드가 있는 문서(들)만 모델에 넘겨주면 된다.

# 작은 조각들로 쪼개어 두면 필요한 것들을 찾기가 용이해진다.
#  - prompt 도 짧아질거다 (적은 token 사용, 적은 비용.)

# split 하는 방법은 다양하다.

In [ ]:
"""
TextSplitter 계층도

BaseDocumentTransformer --> TextSplitter --> <name>TextSplitter  # Example: CharacterTextSplitter
                                             RecursiveCharacterTextSplitter -->  <name>TextSplitter

https://python.langchain.com/api_reference/text_splitters/index.html

"""
None

## RecursiveCharacterTextSplitter

In [18]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

In [22]:
splitter = RecursiveCharacterTextSplitter()

# RecursiveCharacterTextSplitter 는 파일을 split 해주는데
# 문장의 끝이나, 문단의 끝부분마다 끊어준다.
# 문장 중간을 끊지는 않는다.  최대한 문장 중간에서 split 되지 않도록 하려 한다.
# 문장 중간에 짤림으로 의미있는 문장들을 잃고 싶지 않다.

# ↓ splitter 사용방법은 두가지 가 있다.



### 방법1 load(), split_documents()

In [20]:
docs = loader.load()
len(docs)

1

In [23]:
documents = splitter.split_documents(docs)  # -> List[Document]
print(len(documents), '개 Document')
documents

11 개 Document


[Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hat

### 방법2: load_and_split()

In [24]:
documents = loader.load_and_split(text_splitter=splitter) # -> List[Document]
print(len(documents), '개 Document')
documents

11 개 Document


[Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hat

In [26]:
# 첫번째 Document
print(documents[0].page_content)

Part 1, Chapter 1

Part One


1
It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.

The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above his righ

### chunk_size=

In [27]:
# 좀 더 작은 Document 를 만들 필요가 있다.
# 모델의 Context Window 가 크지 않은 경우라든지..
# chunk_size= 값으로 조정해보자

In [28]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, # 얼마나 큰 덩어리로 나눌지 지정.  (아까보다 굉장히 잘개 쪼개질거다)
                        # chunk_size 의 단위는 splitter 마다 다르다.
                    # CharacterTextSplitter 의 경우 chunk_size 는 '문자개수'
)

documents = loader.load_and_split(text_splitter=splitter) # -> List[Document]
print(len(documents), '개 Document')
documents

3675 개 Document


[Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt'}, page_content='Part 1, Chapter 1\n\nPart One'),
 Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt'}, page_content='1'),
 Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt'}, page_content='It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors'),
 Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt'}, page_content='was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of'),
 Document(metadata={'source': 'D:\\MCP2607\\LangWork\\dataset\\files\\chapter_one.txt'}, page_content='cold day in April, and the clocks were st

In [ ]:
# ↑ 문제점: 문단의 중간이 잘려버렸다 -> 문장의 의미가 파괴된다.

# 작은 덩어리이면서 문장의 중간을 잘라먹지 않는 방법은?
# chunk_overlap=
#    split 할때 앞 조각의 일부를 가져와서 연결해준다.
#    Document 간의 겹치는 부분 생길수 있다.

### chunk_overlap=

In [29]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=50,
)

documents = loader.load_and_split(text_splitter=splitter) # -> List[Document]
print(len(documents), '개 Document')

for document in documents[10:15]:
    print('🔷', document.page_content)



247 개 Document
🔷 move. BIG BROTHER IS WATCHING YOU, the caption beneath it ran.
🔷 Inside the flat a fruity voice was reading out a list of figures which had something to do with the production of pig-iron. The voice came from an oblong metal plaque like a dulled mirror which
🔷 an oblong metal plaque like a dulled mirror which formed part of the surface of the right-hand wall. Winston turned a switch and the voice sank somewhat, though the words were still distinguishable.
🔷 though the words were still distinguishable. The instrument (the telescreen, it was called) could be dimmed, but there was no way of shutting it off completely. He moved over to the window: a
🔷 it off completely. He moved over to the window: a smallish, frail figure, the meagreness of his body merely emphasized by the blue overalls which were the uniform of the party. His hair was very


## CharacterTextSplitter

In [30]:
from langchain_text_splitters.character import CharacterTextSplitter